In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [9]:
# --- STEP 1 : Smart Setup & Install Dependencies ---

import subprocess, sys, importlib, os

def safe_pip_install(package, version=None, index_url=None):
    """Try installing; fail gracefully if no network."""
    try:
        cmd = [sys.executable, "-m", "pip", "install", "--quiet"]
        if index_url:
            cmd += ["--index-url", index_url]
        pkg_str = f"{package}=={version}" if version else package
        subprocess.check_call(cmd + [pkg_str])
    except Exception as e:
        print(f"⚠️ Could not install {package}: {e}")
        print("   (Likely no internet; using existing version if available.)")

# --- PyTorch (CPU-only stable build) ---
try:
    import torch
    print(f"Torch already present: {torch.__version__}")
except Exception:
    print("Installing CPU-safe PyTorch 2.4.1 …")
    safe_pip_install("torch", "2.4.1", "https://download.pytorch.org/whl/cpu")
    safe_pip_install("torchvision", "0.19.1", "https://download.pytorch.org/whl/cpu")
    safe_pip_install("torchaudio", "2.4.1", "https://download.pytorch.org/whl/cpu")
    import torch

# --- Core scientific + audio libs ---
for pkg in ["numpy", "pandas", "librosa", "soundfile", "tqdm"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        safe_pip_install(pkg)

# --- Transformers + protobuf (only if torch import succeeded) ---
try:
    import transformers
    print(f"Transformers already present: {transformers.__version__}")
except Exception:
    print("Installing transformers 4.37.2 and protobuf 3.20…")
    safe_pip_install("transformers", "4.37.2")
    safe_pip_install("protobuf", "3.20.*")
    import transformers

# --- Imports & version check ---
import numpy as np, pandas as pd, librosa, soundfile, google.protobuf
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

print("\n--- Environment Summary ---")
print("Torch:", getattr(torch, "__version__", "not found"))
print("Transformers:", getattr(transformers, "__version__", "not found"))
print("Librosa:", librosa.__version__)
print("Protobuf:", google.protobuf.__version__)
print("✅ Environment ready!")


Torch already present: 2.6.0+cu124
Transformers already present: 4.53.3

--- Environment Summary ---
Torch: 2.6.0+cu124
Transformers: 4.53.3
Librosa: 0.11.0
Protobuf: 6.33.0
✅ Environment ready!


In [10]:
# --- STEP 2: Load MELD Dataset CSVs ---
train_df = pd.read_csv("/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/train/train_sent_emo.csv")
val_df   = pd.read_csv("/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/dev_sent_emo.csv")
test_df  = pd.read_csv("/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw/test_sent_emo.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("\nSample:")
print(train_df.head())

Train shape: (9989, 11)
Validation shape: (1109, 11)
Test shape: (2610, 11)

Sample:
   Sr No.                                          Utterance          Speaker  \
0       1  also I was the point person on my companys tr...         Chandler   
1       2                   You mustve had your hands full.  The Interviewer   
2       3                            That I did. That I did.         Chandler   
3       4      So lets talk a little bit about your duties.  The Interviewer   
4       5                             My duties?  All right.         Chandler   

    Emotion Sentiment  Dialogue_ID  Utterance_ID  Season  Episode  \
0   neutral   neutral            0             0       8       21   
1   neutral   neutral            0             1       8       21   
2   neutral   neutral            0             2       8       21   
3   neutral   neutral            0             3       8       21   
4  surprise  positive            0             4       8       21   

      StartTi

In [11]:
# --- STEP 3: Reliable audio feature extraction with live progress (MELD-ready) ---

import os, subprocess, io, librosa, numpy as np, pandas as pd, soundfile as sf, warnings, time
from tqdm import tqdm
warnings.filterwarnings("ignore", category=UserWarning)

# === 1️⃣  Build VideoPath column for MELD ===
train_audio_path = "/kaggle/input/meld-dataset/MELD.Raw/train_splits_audio"
val_audio_path   = "/kaggle/input/meld-dataset/MELD.Raw/dev_splits_complete"
test_audio_path  = "/kaggle/input/meld-dataset/MELD.Raw/output_repeated_splits_test"

def make_path(row, base_path, ext="mp4"):
    return os.path.join(base_path, f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.{ext}")

train_df["VideoPath"] = train_df.apply(lambda r: make_path(r, train_audio_path), axis=1)
val_df["VideoPath"]   = val_df.apply(lambda r: make_path(r, val_audio_path), axis=1)
test_df["VideoPath"]  = test_df.apply(lambda r: make_path(r, test_audio_path), axis=1)

print("✅ Sample training paths:")
print(train_df["VideoPath"].head())

# === 2️⃣  Feature extractor: MFCC + pitch + spectral features ===
def extract_audio_features_fast(video_path):
    """Extract MFCC, pitch, and spectral features from an MP4 file."""
    try:
        if not os.path.exists(video_path):
            return np.zeros(43, dtype=np.float32)

        # Convert video → mono 16 kHz wav in memory
        result = subprocess.run(
            ["ffmpeg", "-i", video_path, "-ac", "1", "-ar", "16000", "-f", "wav", "pipe:1"],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False
        )

        y, sr = sf.read(io.BytesIO(result.stdout), dtype='float32')
        if y is None or len(y) == 0:
            return np.zeros(43, dtype=np.float32)

        # MFCCs (40 × T)
        mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
        # Pitch
        pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
        pitch_mean = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0
        # Spectral features
        centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        rolloff  = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))

        return np.concatenate([mfcc, [pitch_mean, centroid, rolloff]]).astype(np.float32)

    except Exception:
        return np.zeros(43, dtype=np.float32)

# === 3️⃣  Sequential extraction with progress + save ===
def sequential_extract(df, name):
    paths = df["VideoPath"].tolist()
    total = len(paths)
    print(f"\n🚀 Extracting {name} features sequentially ({total} files)...")
    start = time.time()
    results = []

    for p in tqdm(paths, total=total, dynamic_ncols=True):
        results.append(extract_audio_features_fast(p))

    df["audio_features"] = results
    out_name = f"{name}_audio_features_fast.pkl"
    pd.to_pickle(df, out_name)
    print(f"✅ Saved: {out_name} | Time: {(time.time()-start)/60:.1f} min")
    return df

# === 4️⃣  Run for all splits ===
train_df = sequential_extract(train_df, "train")
val_df   = sequential_extract(val_df, "val")
test_df  = sequential_extract(test_df, "test")

print("\n✅ All splits processed successfully!")
print("Example feature vector shape:", train_df["audio_features"].iloc[0].shape)


✅ Sample training paths:
0    /kaggle/input/meld-dataset/MELD.Raw/train_spli...
1    /kaggle/input/meld-dataset/MELD.Raw/train_spli...
2    /kaggle/input/meld-dataset/MELD.Raw/train_spli...
3    /kaggle/input/meld-dataset/MELD.Raw/train_spli...
4    /kaggle/input/meld-dataset/MELD.Raw/train_spli...
Name: VideoPath, dtype: object

🚀 Extracting train features sequentially (9989 files)...


100%|██████████| 9989/9989 [00:00<00:00, 195860.46it/s]


✅ Saved: train_audio_features_fast.pkl | Time: 0.0 min

🚀 Extracting val features sequentially (1109 files)...


100%|██████████| 1109/1109 [00:00<00:00, 159155.65it/s]


✅ Saved: val_audio_features_fast.pkl | Time: 0.0 min

🚀 Extracting test features sequentially (2610 files)...


100%|██████████| 2610/2610 [00:00<00:00, 179959.78it/s]

✅ Saved: test_audio_features_fast.pkl | Time: 0.0 min

✅ All splits processed successfully!
Example feature vector shape: (43,)


In [12]:
!ls /kaggle/input/wav2vec2-base-960h


config.json		       pytorch_model.bin	tokenizer_config.json
feature_extractor_config.json  README.md		vocab.json
model.safetensors	       special_tokens_map.json
preprocessor_config.json       tf_model.h5


In [13]:
# --- STEP 4: Offline Wav2Vec2-base-960h + Low-Level Acoustic Fusion ---

import torch, soundfile as sf, subprocess, io, numpy as np, pandas as pd
from tqdm import tqdm
from transformers.models.wav2vec2 import Wav2Vec2Processor, Wav2Vec2Model


# === 1️⃣ Load the pretrained model from your Kaggle uploaded dataset ===
LOCAL_W2V_PATH = "/kaggle/input/wav2vec2-base-960h"

print("🔹 Loading Wav2Vec2 model from local dataset...")
processor = Wav2Vec2Processor.from_pretrained(LOCAL_W2V_PATH)
model = Wav2Vec2Model.from_pretrained(LOCAL_W2V_PATH)
model.eval()
print("✅ Model loaded successfully from offline path!")

# === 2️⃣ Utility: Extract a 768-D embedding for a single utterance ===
def extract_wav2vec2_embedding(video_path):
    """Convert .mp4 → wav → embedding using Wav2Vec2 (offline-safe)."""
    try:
        # Convert video → mono 16kHz wav bytes (no temp files)
        result = subprocess.run(
            ["ffmpeg", "-i", video_path, "-ac", "1", "-ar", "16000", "-f", "wav", "pipe:1"],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False
        )
        y, sr = sf.read(io.BytesIO(result.stdout), dtype="float32")
        if len(y) == 0:
            return np.zeros(768, dtype=np.float32)

        inputs = processor(y, sampling_rate=sr, return_tensors="pt", padding=True)
        with torch.no_grad():
            emb = model(**inputs).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return emb.astype(np.float32)
    except Exception:
        return np.zeros(768, dtype=np.float32)

# === 3️⃣ Combine the Wav2Vec2 embeddings with your 43-D low-level features ===
def add_embeddings(df, name):
    print(f"\n🚀 Generating Wav2Vec2 embeddings for {name} split ({len(df)} files)...")
    new_embs = []
    for path in tqdm(df["VideoPath"].tolist(), total=len(df), dynamic_ncols=True):
        new_embs.append(extract_wav2vec2_embedding(path))
    df["wav2vec2_emb"] = new_embs

    # Concatenate Wav2Vec2 (768-D) + low-level (43-D)
    if "audio_features" in df.columns:
        combined = [np.concatenate([low, high]) for low, high in zip(df["audio_features"], df["wav2vec2_emb"])]
        df["combined_audio"] = combined
        print(f"✅ Combined feature dimension: {len(df['combined_audio'].iloc[0])}")
    else:
        df["combined_audio"] = df["wav2vec2_emb"]

    out_name = f"{name}_combined_audio.pkl"
    pd.to_pickle(df[["Dialogue_ID", "Utterance_ID", "combined_audio", "Emotion"]], out_name)
    print(f"💾 Saved: {out_name}")
    return df

# === 4️⃣ Load Step 3 outputs ===
train_df = pd.read_pickle("train_audio_features_fast.pkl")
val_df   = pd.read_pickle("val_audio_features_fast.pkl")
test_df  = pd.read_pickle("test_audio_features_fast.pkl")

# === 5️⃣ Run embedding + fusion ===
train_df = add_embeddings(train_df, "train")
val_df   = add_embeddings(val_df, "val")
test_df  = add_embeddings(test_df, "test")

print("\n✅ Step 4 complete — Wav2Vec2 + Low-Level features fused successfully!")
print("Example fused vector length:", len(train_df["combined_audio"].iloc[0]))


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at /kaggle/input/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔹 Loading Wav2Vec2 model from local dataset...
✅ Model loaded successfully from offline path!

🚀 Generating Wav2Vec2 embeddings for train split (9989 files)...


100%|██████████| 9989/9989 [19:04<00:00,  8.73it/s]


✅ Combined feature dimension: 811
💾 Saved: train_combined_audio.pkl

🚀 Generating Wav2Vec2 embeddings for val split (1109 files)...


100%|██████████| 1109/1109 [02:05<00:00,  8.87it/s]


✅ Combined feature dimension: 811
💾 Saved: val_combined_audio.pkl

🚀 Generating Wav2Vec2 embeddings for test split (2610 files)...


100%|██████████| 2610/2610 [05:03<00:00,  8.60it/s]

✅ Combined feature dimension: 811
💾 Saved: test_combined_audio.pkl

✅ Step 4 complete — Wav2Vec2 + Low-Level features fused successfully!
Example fused vector length: 811


In [14]:
# === 3️⃣ Combine the Wav2Vec2 embeddings with your 43-D low-level features ===
def add_embeddings(df, name):
    print(f"\n🚀 Generating Wav2Vec2 embeddings for {name} split ({len(df)} files)...")
    new_embs = []
    for path in tqdm(df["VideoPath"].tolist(), total=len(df), dynamic_ncols=True):
        new_embs.append(extract_wav2vec2_embedding(path))
    df["wav2vec2_emb"] = new_embs

    # Concatenate Wav2Vec2 (768-D) + low-level (43-D)
    if "audio_features" in df.columns:
        combined = [np.concatenate([low, high]) for low, high in zip(df["audio_features"], df["wav2vec2_emb"])]
        df["combined_audio"] = combined
        print(f"✅ Combined feature dimension: {len(df['combined_audio'].iloc[0])}")
    else:
        df["combined_audio"] = df["wav2vec2_emb"]

    out_name = f"{name}_combined_audio.pkl"
    pd.to_pickle(df[["Dialogue_ID", "Utterance_ID", "combined_audio", "Emotion"]], out_name)
    print(f"💾 Saved: {out_name}")
    return df

In [15]:
train_df = add_embeddings(train_df, "train")
val_df   = add_embeddings(val_df, "val")
test_df  = add_embeddings(test_df, "test")



🚀 Generating Wav2Vec2 embeddings for train split (9989 files)...


100%|██████████| 9989/9989 [18:52<00:00,  8.82it/s]


✅ Combined feature dimension: 811
💾 Saved: train_combined_audio.pkl

🚀 Generating Wav2Vec2 embeddings for val split (1109 files)...


100%|██████████| 1109/1109 [02:04<00:00,  8.88it/s]


✅ Combined feature dimension: 811
💾 Saved: val_combined_audio.pkl

🚀 Generating Wav2Vec2 embeddings for test split (2610 files)...


100%|██████████| 2610/2610 [04:53<00:00,  8.88it/s]

✅ Combined feature dimension: 811
💾 Saved: test_combined_audio.pkl


In [17]:
from transformers import BertTokenizer, BertModel

bert_path = "/kaggle/input/bert-base-uncased"   # your uploaded model

text_tokenizer = BertTokenizer.from_pretrained(bert_path)
text_model = BertModel.from_pretrained(bert_path)

print("BERT is loaded correctly!")


BERT is loaded correctly!


In [18]:
# ================================
# STEP 5 : Extract BERT Embeddings
# ================================

import pandas as pd
import numpy as np
from tqdm import tqdm
import torch

# --------------------------------
# Load MELD .csv files (FINAL CORRECT PATHS)
# --------------------------------
base = "/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw"

train_df = pd.read_csv(f"{base}/train/train_sent_emo.csv")
dev_df   = pd.read_csv(f"{base}/dev_sent_emo.csv")
test_df  = pd.read_csv(f"{base}/test_sent_emo.csv")

print("Datasets loaded:")
print("Train:", train_df.shape)
print("Dev:  ", dev_df.shape)
print("Test: ", test_df.shape)

# --------------------------------
# BERT Embedding Function
# --------------------------------
def get_bert_embedding(text):
    encoded = text_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = text_model(**encoded)

    # Mean Pooling → best for emotion recognition
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding

# --------------------------------
# Extraction Function
# --------------------------------
def extract_embeddings(df, split_name):
    embeddings = []

    for text in tqdm(df["Utterance"], desc=f"Extracting BERT for {split_name}"):
        if isinstance(text, float):  # handle NaN
            text = ""

        emb = get_bert_embedding(text)
        embeddings.append(emb)

    embeddings = np.array(embeddings)

    save_path = f"/kaggle/working/bert_embeddings_{split_name}.npy"
    np.save(save_path, embeddings)

    print(f"✔ Saved {split_name} embeddings → {save_path}")
    print("Shape:", embeddings.shape)

    return embeddings

# --------------------------------
# Run Full Extraction
# --------------------------------
train_embeddings = extract_embeddings(train_df, "train")
dev_embeddings   = extract_embeddings(dev_df, "dev")
test_embeddings  = extract_embeddings(test_df, "test")


Datasets loaded:
Train: (9989, 11)
Dev:   (1109, 11)
Test:  (2610, 11)


Extracting BERT for train: 100%|██████████| 9989/9989 [10:41<00:00, 15.57it/s]


✔ Saved train embeddings → /kaggle/working/bert_embeddings_train.npy
Shape: (9989, 768)


Extracting BERT for dev: 100%|██████████| 1109/1109 [01:09<00:00, 15.87it/s]


✔ Saved dev embeddings → /kaggle/working/bert_embeddings_dev.npy
Shape: (1109, 768)


Extracting BERT for test: 100%|██████████| 2610/2610 [02:49<00:00, 15.41it/s]


✔ Saved test embeddings → /kaggle/working/bert_embeddings_test.npy
Shape: (2610, 768)


In [20]:
# --- STEP 5: Optimized Text Embedding Extraction + Multimodal Fusion (FIXED) ---

import torch, numpy as np, pandas as pd
from tqdm import tqdm

# === OFFLINE-SAFE BERT IMPORT (Transformers 4.53.3) ===
try:
    from transformers.models.bert import BertTokenizer, BertModel
except ImportError:
    from transformers import BertTokenizer, BertModel


# === 1️⃣ Load BERT model (offline) ===
LOCAL_BERT_PATH = "/kaggle/input/bert-base-uncased"

print("🔹 Loading BERT model and tokenizer (offline)…")
tokenizer = BertTokenizer.from_pretrained(LOCAL_BERT_PATH, local_files_only=True)
bert_model = BertModel.from_pretrained(LOCAL_BERT_PATH, local_files_only=True)
bert_model.eval()
bert_model.to("cpu")
print("✅ BERT loaded successfully!\n")


# === 2️⃣ Load MELD text CSVs (actual utterances) ===
base = "/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw"

text_train = pd.read_csv(f"{base}/train/train_sent_emo.csv")
text_val   = pd.read_csv(f"{base}/dev_sent_emo.csv")
text_test  = pd.read_csv(f"{base}/test_sent_emo.csv")


# === 3️⃣ Load audio fused embeddings from Step 4 ===
audio_train = pd.read_pickle("train_combined_audio.pkl")
audio_val   = pd.read_pickle("val_combined_audio.pkl")
audio_test  = pd.read_pickle("test_combined_audio.pkl")


# === 4️⃣ Merge text with audio using Dialogue_ID + Utterance_ID ===
def merge_text_audio(text_df, audio_df):
    return text_df.merge(
        audio_df,
        on=["Dialogue_ID", "Utterance_ID"],
        how="inner"
    )

train_df = merge_text_audio(text_train, audio_train)
val_df   = merge_text_audio(text_val, audio_val)
test_df  = merge_text_audio(text_test, audio_test)


# === 5️⃣ Fast batched BERT ===
def get_text_embeddings_batch(text_list, batch_size=32):
    all_embeddings = []

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        )

        with torch.no_grad():
            outputs = bert_model(**inputs)

        batch_emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        all_embeddings.append(batch_emb)

    return np.vstack(all_embeddings).astype(np.float32)


# === 6️⃣ Fuse audio (811) + text (768) ===
def fuse_modalities(df, name):
    print(f"\n🚀 Generating BERT embeddings for {name} split ({len(df)} utterances)…")

    texts = df["Utterance"].fillna("").tolist()

    text_embs = get_text_embeddings_batch(texts, batch_size=32)
    df["text_emb"] = list(text_embs)

    fused = [
        np.concatenate([a, t])
        for a, t in zip(df["combined_audio"], df["text_emb"])
    ]

    df["multimodal_emb"] = fused

    out = f"{name}_multimodal_embeddings.pkl"
    pd.to_pickle(df[["Dialogue_ID", "Utterance_ID", "Emotion", "multimodal_emb"]], out)

    print(f"💾 Saved: {out} | Vector dim = {len(df['multimodal_emb'].iloc[0])}")
    return df


train_df = fuse_modalities(train_df, "train")
val_df   = fuse_modalities(val_df, "val")
test_df  = fuse_modalities(test_df, "test")

print("\n🎉 Step 5 COMPLETE — Multimodal fusion successful!")
print("Vector length example:", len(train_df["multimodal_emb"].iloc[0]))


🔹 Loading BERT model and tokenizer (offline)…
✅ BERT loaded successfully!


🚀 Generating BERT embeddings for train split (9989 utterances)…


KeyboardInterrupt: 

In [22]:
import pandas as pd
import numpy as np

# 1. Load audio vectors
audio_train = pd.read_pickle("train_combined_audio.pkl")
audio_val   = pd.read_pickle("val_combined_audio.pkl")
audio_test  = pd.read_pickle("test_combined_audio.pkl")

# 2. Load text embeddings (same order as MELD CSV)
text_train_emb = np.load("/kaggle/working/bert_embeddings_train.npy")
text_val_emb   = np.load("/kaggle/working/bert_embeddings_dev.npy")
text_test_emb  = np.load("/kaggle/working/bert_embeddings_test.npy")

# 3. Load MELD CSVs to attach text to the correct utterances
base = "/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw"

text_train_df = pd.read_csv(f"{base}/train/train_sent_emo.csv")
text_val_df   = pd.read_csv(f"{base}/dev_sent_emo.csv")
text_test_df  = pd.read_csv(f"{base}/test_sent_emo.csv")

# 4. Add BERT embeddings to text dfs
text_train_df["text_emb"] = list(text_train_emb)
text_val_df["text_emb"]   = list(text_val_emb)
text_test_df["text_emb"]  = list(text_test_emb)


In [25]:
def merge_text_audio(text_df, audio_df):
    return text_df.merge(
        audio_df,
        on=["Dialogue_ID", "Utterance_ID"],
        how="inner"
    )

train_merged = merge_text_audio(text_train_df, audio_train)
val_merged   = merge_text_audio(text_val_df, audio_val)
test_merged  = merge_text_audio(text_test_df, audio_test)


In [29]:
# FIX: Ensure the Emotion column exists with correct name
for df in [train_merged, val_merged, test_merged]:
    # If Emotion from text side exists
    if "Emotion_x" in df.columns:
        df.rename(columns={"Emotion_x": "Emotion"}, inplace=True)
    # If Emotion from audio side exists
    if "Emotion_y" in df.columns:
        df.rename(columns={"Emotion_y": "Emotion"}, inplace=True)
    # If Emotion missing entirely → fallback to text side
    if "Emotion" not in df.columns and "Emotion" in text_train_df.columns:
        df["Emotion"] = df["Emotion"]


In [30]:
def fuse(df, name):
    fused = []
    for audio, text in zip(df["combined_audio"], df["text_emb"]):
        fused.append(np.concatenate([audio, text]))

    df["multimodal_emb"] = fused

    out = f"{name}_multimodal_embeddings.pkl"
    df[["Dialogue_ID", "Utterance_ID", "Emotion", "multimodal_emb"]].to_pickle(out)
    print(f"Saved: {out} | dim = {len(df['multimodal_emb'].iloc[0])}")
    return df

train_fused = fuse(train_merged, "train")
val_fused   = fuse(val_merged, "val")
test_fused  = fuse(test_merged, "test")


Saved: train_multimodal_embeddings.pkl | dim = 1579
Saved: val_multimodal_embeddings.pkl | dim = 1579
Saved: test_multimodal_embeddings.pkl | dim = 1579


In [39]:
print(train_df.columns)
print("\n")
print(train_df.head())


Index(['Dialogue_ID', 'Utterance_ID', 'Emotion', 'Emotion', 'multimodal_emb'], dtype='object')


   Dialogue_ID  Utterance_ID   Emotion   Emotion  \
0            0             0   neutral   neutral   
1            0             1   neutral   neutral   
2            0             2   neutral   neutral   
3            0             3   neutral   neutral   
4            0             4  surprise  surprise   

                                      multimodal_emb  
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
1  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
2  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
3  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
4  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  


In [45]:
# =============================================
# FIX EMOTION COLUMN FOR YOUR EXACT DATAFRAME
# =============================================

def clean_emotion_columns(df):

    # Drop ALL Emotion_x columns (duplicates)
    drop_cols = [c for c in df.columns if c.startswith("Emotion_x")]
    if len(drop_cols) > 0:
        df.drop(columns=drop_cols, inplace=True)

    # If Emotion_y exists → rename it to Emotion
    if "Emotion_y" in df.columns:
        df.rename(columns={"Emotion_y": "Emotion"}, inplace=True)

    # Final sanity check: ensure we have Emotion
    if "Emotion" not in df.columns:
        raise ValueError("❌ Emotion column is still missing after cleanup!")

    return df


In [46]:
# ============================
# FINAL STEP 9 
# ============================

import pandas as pd
import numpy as np
import torch

# 1. Load fused multimodal files
train_df = pd.read_pickle("train_multimodal_embeddings.pkl")
val_df   = pd.read_pickle("val_multimodal_embeddings.pkl")
test_df  = pd.read_pickle("test_multimodal_embeddings.pkl")

# 2. Restore Emotion using MELD CSV
base = "/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw"

text_train = pd.read_csv(f"{base}/train/train_sent_emo.csv")
text_val   = pd.read_csv(f"{base}/dev_sent_emo.csv")
text_test  = pd.read_csv(f"{base}/test_sent_emo.csv")

train_df = train_df.merge(text_train[["Dialogue_ID","Utterance_ID","Emotion"]],
                          on=["Dialogue_ID","Utterance_ID"],
                          how="left")
val_df = val_df.merge(text_val[["Dialogue_ID","Utterance_ID","Emotion"]],
                      on=["Dialogue_ID","Utterance_ID"],
                      how="left")
test_df = test_df.merge(text_test[["Dialogue_ID","Utterance_ID","Emotion"]],
                        on=["Dialogue_ID","Utterance_ID"],
                        how="left")

# 3. Clean duplicate Emotion columns
train_df = clean_emotion_columns(train_df)
val_df   = clean_emotion_columns(val_df)
test_df  = clean_emotion_columns(test_df)

print("COLUMNS AFTER FIX (train_df):", train_df.columns)

# ---------------------------------------------------------
# 4. Convert embeddings
# ---------------------------------------------------------
X_train = np.stack(train_df["multimodal_emb"].values)
X_val   = np.stack(val_df["multimodal_emb"].values)
X_test  = np.stack(test_df["multimodal_emb"].values)

# ---------------------------------------------------------
# 5. Build label map
# ---------------------------------------------------------
emotions = sorted(train_df["Emotion"].unique())
label_map = {e: i for i, e in enumerate(emotions)}

# ---------------------------------------------------------
# 6. Convert labels
# ---------------------------------------------------------
y_train = train_df["Emotion"].map(label_map).values
y_val   = val_df["Emotion"].map(label_map).values
y_test  = test_df["Emotion"].map(label_map).values

print("\nShapes:")
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "| y_val:", y_val.shape)
print("X_test: ", X_test.shape,  "| y_test:", y_test.shape)

print("\nEmotion label map:", label_map)


COLUMNS AFTER FIX (train_df): Index(['Dialogue_ID', 'Utterance_ID', 'multimodal_emb', 'Emotion'], dtype='object')

Shapes:
X_train: (9989, 1579) | y_train: (9989,)
X_val:   (1109, 1579) | y_val: (1109,)
X_test:  (2610, 1579) | y_test: (2610,)

Emotion label map: {'anger': 0, 'disgust': 1, 'fear': 2, 'joy': 3, 'neutral': 4, 'sadness': 5, 'surprise': 6}


In [47]:
# ============================
# FINAL STEP 10
# Split audio/text + create DataLoaders
# ============================

import torch
from torch.utils.data import TensorDataset, DataLoader

# Fused vector = 1579 dims:
#  → Audio = first 811 dims
#  → Text  = remaining 768 dims

# 1. Split multimodal vector
X_train_audio = torch.tensor([x[:811]  for x in X_train], dtype=torch.float32)
X_train_text  = torch.tensor([x[811:] for x in X_train], dtype=torch.float32)

X_val_audio = torch.tensor([x[:811]  for x in X_val], dtype=torch.float32)
X_val_text  = torch.tensor([x[811:] for x in X_val], dtype=torch.float32)

X_test_audio = torch.tensor([x[:811]  for x in X_test], dtype=torch.float32)
X_test_text  = torch.tensor([x[811:] for x in X_test], dtype=torch.float32)

# 2. Labels
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t   = torch.tensor(y_val,   dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

# 3. Build datasets
train_ds = TensorDataset(X_train_audio, X_train_text, y_train_t)
val_ds   = TensorDataset(X_val_audio,   X_val_text,   y_val_t)
test_ds  = TensorDataset(X_test_audio,  X_test_text,  y_test_t)

# 4. Build loaders
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)
test_loader  = DataLoader(test_ds, batch_size=32)

print("DataLoaders ready!")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))


DataLoaders ready!
Train batches: 313
Val batches: 35
Test batches: 82


In [50]:
# ============================
# STEP 11
# Cross-Modal Attention Fusion Model + Training Loop
# ============================

import torch
import torch.nn as nn
from torch.optim import Adam

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ---------------------------------------
# 1. Define Cross-Modal Attention Network
# ---------------------------------------
class CrossModalAttentionFusion(nn.Module):
    def __init__(self, audio_dim=811, text_dim=768, hidden_dim=256, num_classes=7):
        super().__init__()

        # Project audio & text into same hidden space
        self.audio_proj = nn.Linear(audio_dim, hidden_dim)
        self.text_proj = nn.Linear(text_dim, hidden_dim)

        # Cross-modal attention: audio attends to text
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=4,
            batch_first=True
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, audio_vec, text_vec):
        # (B, hidden)
        A = self.audio_proj(audio_vec).unsqueeze(1)  # (B,1,H)
        T = self.text_proj(text_vec).unsqueeze(1)    # (B,1,H)

        # Cross-modal attention (audio attends to text)
        attn_output, _ = self.attn(A, T, T)  # Query=A, Key=T, Value=T

        # Concatenate attention output + projected text
        fused = torch.cat([attn_output.squeeze(1), T.squeeze(1)], dim=1)

        return self.classifier(fused)

# ---------------------------------------
# 2. Initialize model, loss, optimizer
# ---------------------------------------
model = CrossModalAttentionFusion().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-4)

# ---------------------------------------
# 3. Training Loop
# ---------------------------------------
EPOCHS = 35

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for audio_batch, text_batch, labels in train_loader:
        audio_batch = audio_batch.to(device)
        text_batch = text_batch.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(audio_batch, text_batch)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Training Loss: {avg_loss:.4f}")

    # -----------------------------------
    # Validation
    # -----------------------------------
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for audio_batch, text_batch, labels in val_loader:
            audio_batch = audio_batch.to(device)
            text_batch = text_batch.to(device)
            labels = labels.to(device)

            outputs = model(audio_batch, text_batch)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += len(labels)

    val_acc = correct / total
    print(f"  Validation Accuracy: {val_acc:.4f}")


Using device: cpu
Epoch 1/35 | Training Loss: 1.3363
  Validation Accuracy: 0.5654
Epoch 2/35 | Training Loss: 1.1412
  Validation Accuracy: 0.5987
Epoch 3/35 | Training Loss: 1.0980
  Validation Accuracy: 0.5960
Epoch 4/35 | Training Loss: 1.0817
  Validation Accuracy: 0.5897
Epoch 5/35 | Training Loss: 1.0658
  Validation Accuracy: 0.5933
Epoch 6/35 | Training Loss: 1.0511
  Validation Accuracy: 0.5924
Epoch 7/35 | Training Loss: 1.0396
  Validation Accuracy: 0.5933
Epoch 8/35 | Training Loss: 1.0282
  Validation Accuracy: 0.5987
Epoch 9/35 | Training Loss: 1.0130
  Validation Accuracy: 0.5906
Epoch 10/35 | Training Loss: 1.0143
  Validation Accuracy: 0.5942
Epoch 11/35 | Training Loss: 1.0018
  Validation Accuracy: 0.5843
Epoch 12/35 | Training Loss: 0.9901
  Validation Accuracy: 0.6005
Epoch 13/35 | Training Loss: 0.9819
  Validation Accuracy: 0.6032
Epoch 14/35 | Training Loss: 0.9748
  Validation Accuracy: 0.5951
Epoch 15/35 | Training Loss: 0.9712
  Validation Accuracy: 0.5951
E

In [51]:
# ============================
# STEP 11 (IMPROVED MODEL)
# Cross-modal Attention v2 + Early Stopping
# ============================

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# --------------------------------------------------------
# 1. Improved Cross-Modal Attention Fusion Model (v2)
# --------------------------------------------------------
class CrossModalAttentionFusionV2(nn.Module):
    def __init__(self, audio_dim=811, text_dim=768, hidden_dim=512, num_classes=7):
        super().__init__()

        # Project audio & text into same representation space
        self.audio_proj = nn.Linear(audio_dim, hidden_dim)
        self.text_proj = nn.Linear(text_dim, hidden_dim)

        # Two-layer cross-modal attention blocks
        self.attn1 = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)
        self.attn2 = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.drop = nn.Dropout(0.3)
        self.act = nn.GELU()

        # Final fusion + classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, audio_vec, text_vec):

        A = self.audio_proj(audio_vec).unsqueeze(1)  # (B,1,H)
        T = self.text_proj(text_vec).unsqueeze(1)    # (B,1,H)

        # ----- Cross-modal attention Layer 1 -----
        # Audio attends to text
        attn_out1, _ = self.attn1(A, T, T)
        A1 = self.norm1(A + self.drop(attn_out1))

        # Text attends to audio (bidirectional attention)
        attn_out1b, _ = self.attn1(T, A, A)
        T1 = self.norm1(T + self.drop(attn_out1b))

        # ----- Cross-modal attention Layer 2 -----
        attn_out2, _ = self.attn2(A1, T1, T1)
        A2 = self.norm2(A1 + self.drop(attn_out2))

        attn_out2b, _ = self.attn2(T1, A1, A1)
        T2 = self.norm2(T1 + self.drop(attn_out2b))

        # Final fused representation
        fused = torch.cat([A2.squeeze(1), T2.squeeze(1)], dim=1)

        return self.classifier(fused)

# --------------------------------------------------------
# 2. Initialize model + optimizer + scheduler
# --------------------------------------------------------
model = CrossModalAttentionFusionV2().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)

# --------------------------------------------------------
# 3. Early Stopping Utility
# --------------------------------------------------------
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_acc = 0
        self.counter = 0
        self.best_state = None

    def step(self, acc, model):
        if acc > self.best_acc:
            self.best_acc = acc
            self.best_state = model.state_dict()
            self.counter = 0
        else:
            self.counter += 1

        return self.counter >= self.patience


early_stop = EarlyStopping(patience=7)

# --------------------------------------------------------
# 4. Training Loop with Early Stopping
# --------------------------------------------------------
EPOCHS = 60  # Train up to 60, but early stopping will stop sooner

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for audio_batch, text_batch, labels in train_loader:
        audio_batch = audio_batch.to(device)
        text_batch = text_batch.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(audio_batch, text_batch)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # ---------------------
    # Validation
    # ---------------------
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for audio_batch, text_batch, labels in val_loader:
            audio_batch = audio_batch.to(device)
            text_batch = text_batch.to(device)
            labels = labels.to(device)

            outputs = model(audio_batch, text_batch)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += len(labels)

    val_acc = correct / total
    scheduler.step(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Check early stopping
    if early_stop.step(val_acc, model):
        print(f"\n⏹ Early stopping at epoch {epoch+1}")
        break

# --------------------------------------------------------
# 5. Restore Best Model
# --------------------------------------------------------
model.load_state_dict(early_stop.best_state)
print("\n⭐ Best Validation Accuracy:", early_stop.best_acc)


Using device: cpu
Epoch 1/60 | Loss: 1.2695 | Val Acc: 0.5906
Epoch 2/60 | Loss: 1.1248 | Val Acc: 0.5780
Epoch 3/60 | Loss: 1.0856 | Val Acc: 0.6050
Epoch 4/60 | Loss: 1.0607 | Val Acc: 0.5888
Epoch 5/60 | Loss: 1.0366 | Val Acc: 0.5969
Epoch 6/60 | Loss: 1.0150 | Val Acc: 0.5798
Epoch 7/60 | Loss: 0.9910 | Val Acc: 0.5960
Epoch 8/60 | Loss: 0.9365 | Val Acc: 0.6041
Epoch 9/60 | Loss: 0.9217 | Val Acc: 0.6023
Epoch 10/60 | Loss: 0.9035 | Val Acc: 0.6023

⏹ Early stopping at epoch 10

⭐ Best Validation Accuracy: 0.6050495942290351


In [53]:
# ============================
# STEP 12 — Test Evaluation
# ============================

import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for audio_batch, text_batch, labels in test_loader:
        audio_batch = audio_batch.to(device)
        text_batch = text_batch.to(device)
        labels = labels.to(device)

        outputs = model(audio_batch, text_batch)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# ---------------------------
# 1. Test Accuracy
# ---------------------------
test_acc = accuracy_score(all_labels, all_preds)
print(" TEST ACCURACY:", round(test_acc, 4))

# ---------------------------
# 2. Classification Report
# ---------------------------
print("\n CLASSIFICATION REPORT:")
print(classification_report(all_labels, all_preds, target_names=emotions))

# ---------------------------
# 3. Confusion Matrix
# ---------------------------
print("\n CONFUSION MATRIX:")
print(confusion_matrix(all_labels, all_preds))


 TEST ACCURACY: 0.6268

 CLASSIFICATION REPORT:
              precision    recall  f1-score   support

       anger       0.52      0.34      0.41       345
     disgust       0.45      0.07      0.13        68
        fear       0.19      0.08      0.11        50
         joy       0.55      0.54      0.55       402
     neutral       0.71      0.85      0.78      1256
     sadness       0.41      0.23      0.29       208
    surprise       0.50      0.61      0.55       281

    accuracy                           0.63      2610
   macro avg       0.48      0.39      0.40      2610
weighted avg       0.60      0.63      0.60      2610


 CONFUSION MATRIX:
[[ 117    1    1   51  110    8   57]
 [  12    5    1    4   34    2   10]
 [   5    0    4    7   19    6    9]
 [  26    0    2  219  119    4   32]
 [  25    2    7   63 1073   45   41]
 [  20    1    5   20   93   47   22]
 [  20    2    1   32   53    2  171]]


In [54]:
# ============================
# STEP 13 — SAVE YOUR MODEL
# ============================

# 1. Save model weights
torch.save(model.state_dict(), "multimodal_emotion_model.pt")

# 2. Save label map
import json
with open("label_map.json", "w") as f:
    json.dump(label_map, f)

print("Model and label map saved successfully!")


Model and label map saved successfully!


In [55]:
# ============================
# STEP 14 — LOAD SAVED MODEL
# ============================

import torch
import json

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load label map
with open("label_map.json", "r") as f:
    label_map = json.load(f)

# Reverse map
idx_to_label = {v: k for k, v in label_map.items()}

# Recreate the model architecture
model = CrossModalAttentionFusionV2().to(device)  # if this was your final trained model

# Load weights
model.load_state_dict(torch.load("multimodal_emotion_model.pt", map_location=device))
model.eval()

print("Model loaded successfully!")


Using device: cpu
Model loaded successfully!


In [56]:
def predict_emotion(model, audio_vec, text_vec):
    model.eval()
    with torch.no_grad():
        audio_t = torch.tensor(audio_vec, dtype=torch.float32).unsqueeze(0)
        text_t  = torch.tensor(text_vec,  dtype=torch.float32).unsqueeze(0)

        logits = model(audio_t, text_t)
        pred = logits.argmax(dim=1).item()

    return idx_to_label[pred]


In [58]:
from transformers import BertTokenizer, BertModel, Wav2Vec2Processor, Wav2Vec2Model
import numpy as np
import torch, soundfile as sf, subprocess, io

# Load BERT (offline)
LOCAL_BERT = "/kaggle/input/bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(LOCAL_BERT)
bert_model = BertModel.from_pretrained(LOCAL_BERT).to(device)
bert_model.eval()

# Load Wav2Vec2 (offline)
LOCAL_W2V = "/kaggle/input/wav2vec2-base-960h"
processor = Wav2Vec2Processor.from_pretrained(LOCAL_W2V)
wav2vec_model = Wav2Vec2Model.from_pretrained(LOCAL_W2V).to(device)
wav2vec_model.eval()


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at /kaggle/input/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wav2Vec2Model(
  (feature_extractor): Wav2Vec2FeatureEncoder(
    (conv_layers): ModuleList(
      (0): Wav2Vec2GroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): Wav2Vec2FeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Wav2Vec2Encoder(
    (pos_conv_embed): Wav2Vec2PositionalConvEmbedding(
  

In [60]:
def get_text_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        outputs = bert_model(**inputs.to(device))
    emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return emb.astype(np.float32)


In [68]:
def get_audio_embedding(audio_path):
    try:
        # Convert any audio/video to WAV bytes
        result = subprocess.run(
            ["ffmpeg", "-i", audio_path, "-ac", "1", "-ar", "16000", "-f", "wav", "pipe:1"],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL
        )

        if result.stdout is None or len(result.stdout) == 0:
            # audio not readable → return zero vector
            return np.zeros(811, dtype=np.float32)

        y, sr = sf.read(io.BytesIO(result.stdout), dtype="float32")

        if len(y) == 0:
            return np.zeros(811, dtype=np.float32)

        inputs = processor(y, sampling_rate=sr, return_tensors="pt", padding=True)
        with torch.no_grad():
            out = wav2vec_model(**inputs.to(device))
            emb = out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

        return emb.astype(np.float32)

    except Exception:
        # ANY ffmpeg / soundfile error → return safe zero vector
        return np.zeros(811, dtype=np.float32)


In [69]:
def build_multimodal_vector(text, audio_path):
    text_emb = get_text_embedding(text)        # 768D
    audio_emb = get_audio_embedding(audio_path) # 811D
    fused = np.concatenate([audio_emb, text_emb])  # 1579D
    return fused


In [70]:
def predict_emotion(model, audio_path, text):
    vec = build_multimodal_vector(text, audio_path)   # 1579D
    audio = torch.tensor(vec[:811], dtype=torch.float32).unsqueeze(0).to(device)
    text  = torch.tensor(vec[811:], dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(audio, text)
        pred = logits.argmax(dim=1).item()
    
    return idx_to_label[pred]


In [71]:
emotion = predict_emotion(
    model,
    audio_path="/kaggle/input/myfile.wav",
    text="I can’t believe you did this!"
)

print("Predicted Emotion:", emotion)


Predicted Emotion: surprise
